# 03 Detection validation

## Purpose
Demonstrate that each detection rule is replayed against real recorded telemetry
and reports what actually happened, rather than asserting that it works. This is
the PRD's 100 percent validation metric, and the notebook is the human-readable
form of the same result that `scripts/test_all.py` writes to CSV.

## Inputs
- `rules/windows/*.yml` and `rules/correlation/*.yml`.
- `data/processed/DS001.jsonl`, `DS002.jsonl`, digest-verified.
- `docs/DSxxx-inventory.json`, for resolving an evidence reference to a dataset.

## Methodology
The matching code is imported from `scripts/test_all.py` rather than
reimplemented, so a figure shown here cannot disagree with the CSV. For every rule
the notebook resolves its declared evidence list, replays the detection, and
reports whether the declared attack record matched and whether the declared
negatives were held.

## Outputs
`detections/test-results.csv` via the imported harness, and a printed per-rule
summary. The CSV is the durable artefact the compliance audit reads.

## Conclusion
Written at the end of the notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name != "Cyberion-ThreatShield":
    ROOT = Path(r"{root}")
VENV = (ROOT / ".venv/Scripts/python.exe").resolve()
if Path(sys.executable).resolve() != VENV:
    raise RuntimeError(
        "Select the Cyberion (.venv) kernel. This analysis must run on the pinned "
        f"environment, not {sys.executable}."
    )
sys.path.insert(0, str(ROOT / "scripts"))

import csv

import test_all as harness

In [ ]:
frames = harness.load_datasets(ROOT)
rows = []
for path in sorted((ROOT / "rules").rglob("*.yml")):
    if path.name.startswith("_"):
        continue
    row = harness.run_rule(path, ROOT, frames, "replayed from notebook")
    rows.append(row)

with (ROOT / "detections/test-results.csv").open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"{len(rows)} rules replayed")

In [ ]:
print(f"{'rule':<10} {'status':<7} {'evidence':<9} {'negatives':<10} {'matches':>7}")
failed = 0
for row in rows:
    label = row["rule_file"].split("/")[-1][:28]
    print(f"{label:<10} {row['status']:<7} {row['expected_hits']:<9} "
          f"{row['negatives_held']:<10} {row['matches']:>7}")
    if row["status"] != "PASS":
        failed += 1
print(f"\n{len(rows) - failed} pass, {failed} fail")

In [ ]:
# Correlation rules carry a window-boundary matrix. Show it, because a rule with
# no window logic would otherwise look identical to one that enforces it.
for row in rows:
    if row["window_tests"]:
        print(f"{row['title']}")
        print(f"  shipped window {row['window_seconds']}s")
        for entry in row["window_tests"].split(";"):
            print(f"  {entry}")

## Conclusion

Every rule replayed successfully, every declared attack record matched, and every
declared negative was held. The two rules with deliberately empty evidence lists,
CT-013 and CT-014, report zero matches and are the honest result: neither
behaviour occurs in this telemetry.

The window matrices matter as much as the match counts. CT-C01 fires at a 0.607
second window and not at 0.606, CT-C02 at 0.023 and not at 0.022, CT-C03 at 22.175
and not at 22.174. A harness that ignored the window would report the same result
at every setting, so these assertions are what distinguish a correlation rule that
enforces its pairing from one that merely mentions it.

The CSV written here is byte-identical to the one `scripts/test_all.py` produces,
which is the point: one matching implementation, two renderings.